# Meridian — Sharpe Head Training (Colab)

Trains the `MeridianS` Sharpe-optimising head on top of the trained return head.
Checkpoints save to Google Drive every 5000 steps. Re-run all cells to resume after a session drop.

**Requires `colab_portfolio_return.ipynb` to have completed first** — `return_head_best.pt` must exist in Drive.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
HF_USERNAME   = 'Verdenroz'
HF_TOKEN      = ''            # HF token with read access (required if repos are private)
                              # create at: huggingface.co/settings/tokens
HF_MODEL_REPO = f'{HF_USERNAME}/meridian-checkpoints'
HF_DATA_REPO  = f'{HF_USERNAME}/meridian-data'

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/meridian'
os.makedirs(f'{DRIVE_DIR}/checkpoints/portfolio', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/mmaps',            exist_ok=True)

# Verify return head checkpoint exists before proceeding
return_best = f'{DRIVE_DIR}/checkpoints/portfolio/return_head_best.pt'
assert os.path.exists(return_best), (
    f'return_head_best.pt not found in Drive.\n'
    f'Complete colab_portfolio_return.ipynb first, or copy the checkpoint to:\n'
    f'  {return_best}'
)
print('Drive ready:', DRIVE_DIR)
print('Return head checkpoint found.')

In [ ]:
# ── 2. HuggingFace login + install deps ───────────────────────────────────────
import subprocess, sys
subprocess.run([
    'pip', 'install', '-q',
    'numpy>=2.1', 'pandas>=2.2', 'pyyaml>=6.0',
    'hmmlearn>=0.3', 'faiss-cpu>=1.8', 'tokenizers>=0.20',
    'huggingface_hub',
], check=True)

from huggingface_hub import login, snapshot_download
if HF_TOKEN:
    login(token=HF_TOKEN)
print('HuggingFace ready.')

In [ ]:
# ── 3. Download training code + symbols file ─────────────────────────────────
import os, shutil
from huggingface_hub import hf_hub_download

if not os.path.exists('/content/meridian'):
    local = snapshot_download(repo_id=HF_MODEL_REPO, allow_patterns='code/**')
    shutil.copytree(os.path.join(local, 'code'), '/content/meridian')
    print('Code downloaded.')
else:
    print('Code already present.')

# symbols.txt lives at repo root, not inside code/, so fetch it separately.
symbols_path = '/content/meridian/symbols.txt'
if not os.path.exists(symbols_path):
    path = hf_hub_download(repo_id=HF_MODEL_REPO, filename='symbols.txt')
    shutil.copy(path, symbols_path)
    print(f'symbols.txt downloaded → {symbols_path}')
else:
    print('symbols.txt already present.')

In [ ]:
# ── 4. Download data (cached to Drive across sessions) ────────────────────────
import os, shutil

data_dir  = f'{DRIVE_DIR}/data/mmaps'
train_dir = f'{data_dir}/train'
has_data  = os.path.exists(train_dir) and any(f.endswith('.f32bin') for f in os.listdir(train_dir))

if not has_data:
    snapshot_download(repo_id=HF_DATA_REPO, repo_type='dataset', local_dir=data_dir)
    # If data was uploaded with a path prefix, snapshot_download nests it — flatten.
    for prefix in [os.path.join(data_dir, 'data', 'mmaps'), os.path.join(data_dir, 'mmaps')]:
        if os.path.exists(os.path.join(prefix, 'train')):
            for item in os.listdir(prefix):
                src = os.path.join(prefix, item)
                dst = os.path.join(data_dir, item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(os.path.join(data_dir, prefix.replace(data_dir + '/', '').split('/')[0]), ignore_errors=True)
            break
    print(f'Data downloaded → {data_dir}')
else:
    print('Data already in Drive.')

In [ ]:
# ── 5. Copy data to local SSD for fast I/O ───────────────────────────────────────────────
import shutil, os

local_data = '/content/data/mmaps'
if not os.path.exists(f'{local_data}/train'):
    print('Copying data from Drive to local SSD (one-time per session)...')
    shutil.copytree(f'{DRIVE_DIR}/data/mmaps', local_data)
    print('Done.')
else:
    print('Local data already present.')

train_files = [f for f in os.listdir(f'{local_data}/train') if f.endswith('.f32bin')]
real_shards  = sorted(f for f in train_files if not f.startswith('FDA_SYNTH_'))
synth_shards = sorted(f for f in train_files if f.startswith('FDA_SYNTH_'))
print(f'Real train shards: {len(real_shards)}  |  FDA synth shards: {len(synth_shards)}')
if not synth_shards:
    print('WARNING: no FDA shards detected — training will run real-data-only '
          '(spec §11.4c expects ∼50k synthetic windows). Run colab_fda.ipynb first.')

In [ ]:
# ── 6. Auto-detect GPU ────────────────────────────────────────────────────────
import torch, os
assert torch.cuda.is_available(), 'No GPU — enable it in Runtime → Change runtime type.'

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  ({vram_gb:.0f} GB VRAM)')

# TF32 matmul — free speedup on Ampere+ with negligible numerical impact.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# See colab_portfolio_return.ipynb cell-gpu for the memory model rationale.
PTA_DETERMINISTIC = True
DISABLE_REGIME_BALANCE = False
WARMUP_BATCH_SIZE = 8
USE_COMPILE = True
REGIME_POOL_SIZE = 64

if vram_gb >= 75:
    BATCH_SIZE    = 4
    BPTT_CHUNK    = 16
    PTA_MICRO_BS  = 10
elif vram_gb >= 35:
    BATCH_SIZE    = 4
    BPTT_CHUNK    = 8
    PTA_MICRO_BS  = 4
elif vram_gb >= 22:
    BATCH_SIZE    = 4
    BPTT_CHUNK    = 4
    PTA_MICRO_BS  = 2
else:
    BATCH_SIZE    = 2
    BPTT_CHUNK    = 2
    PTA_MICRO_BS  = 1
    DISABLE_REGIME_BALANCE = True
    print('WARNING: 16GB tier — dropping to B=2 and disabling regime-balanced sampling (spec deviation).')

WORKERS = min(8, os.cpu_count() or 1)
print(
    f'batch={BATCH_SIZE}  warmup_batch={WARMUP_BATCH_SIZE}  bptt_chunk={BPTT_CHUNK}  '
    f'pta_micro_bs={PTA_MICRO_BS}  pta_deterministic={PTA_DETERMINISTIC}  '
    f'regime_pool={REGIME_POOL_SIZE}  compile={USE_COMPILE}  '
    f'regime_balance={not DISABLE_REGIME_BALANCE}  workers={WORKERS}'
)

In [ ]:
# ── 7. Train Sharpe head (auto-resumes from Drive) ────────────────────────────
import os, sys, subprocess

DEBUG = True

CKPT_PORT   = f'{DRIVE_DIR}/checkpoints/portfolio'
return_best = f'{CKPT_PORT}/return_head_best.pt'
sharpe_best = f'{CKPT_PORT}/sharpe_head_best.pt'
warmup_ckpt = f'{CKPT_PORT}/sharpe_cross_asset_warmup.pt'

import re, glob

def _latest_step_ckpt(ckpt_dir, head):
    candidates = []
    for path in glob.glob(os.path.join(ckpt_dir, f'{head}_head_step*.pt')):
        m = re.search(rf'{head}_head_step(\d+)\.pt$', path)
        if m:
            candidates.append((int(m.group(1)), path))
    return max(candidates, key=lambda x: x[0])[1] if candidates else None

# Resume priority: most recent step checkpoint first, then best, then warmup,
# then the trained return head (Sharpe head is trained on top of the return head).
latest_step = _latest_step_ckpt(CKPT_PORT, 'sharpe')
if latest_step is not None:
    resume_from = latest_step
    print(f'Resuming from latest step checkpoint: {resume_from}')
elif os.path.exists(sharpe_best):
    resume_from = sharpe_best
    print(f'Resuming from best-val checkpoint: {resume_from}')
elif os.path.exists(warmup_ckpt):
    resume_from = warmup_ckpt
    print(f'Resuming from cross-asset warmup: {resume_from}')
else:
    resume_from = return_best
    print(f'Cold start from trained return head: {resume_from}')

cmd = [
    sys.executable, '-u',
    '-m', 'train.portfolio_train',
    '--head',                          'sharpe',
    '--data-dir',                      '/content/data/mmaps',
    '--symbols-file',                  '/content/meridian/symbols.txt',
    '--steps',                         '30000',
    '--batch-size',                    str(BATCH_SIZE),
    '--cross-asset-warmup-batch-size', str(WARMUP_BATCH_SIZE),
    '--seq-len',                       '64',
    '--lr',                            '3e-5',
    '--log-every',                     '10',
    '--eval-every',                    '250',
    '--save-every',                    '250',
    '--bptt-chunk',                    str(BPTT_CHUNK),
    '--pta-micro-batch-size',          str(PTA_MICRO_BS),
    '--regime-pool-size',              str(REGIME_POOL_SIZE),
    '--checkpoint-dir',                CKPT_PORT,
    '--resume-backbone',               resume_from,
    '--workers',                       str(WORKERS),
]
if PTA_DETERMINISTIC:
    cmd.append('--pta-deterministic')
if USE_COMPILE:
    cmd.append('--compile')
if DISABLE_REGIME_BALANCE:
    cmd.append('--disable-regime-balance')
if DEBUG:
    cmd.append('--debug')

env = os.environ.copy()
env['PYTHONPATH']         = '/content/meridian'
env['PYTHONUNBUFFERED']   = '1'
env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

print('Running:', ' '.join(cmd))
print('-' * 80)

process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Training failed with return code {process.returncode}')
print('Training completed.')